In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline


# **Data quality**

1.Aggregate rows (e.g. "World") have no ISO "Code" value, so we must NOT
  filter on non-null codes,that would silently drop the rows we need.

2.The six files have slightly different row counts / year ranges at country
  level, so an inner join on Year is used to avoid introducing missing values.

3."World" is already a pre-aggregated row in each source file, so we use it
  directly rather than re-summing all countries ourselves (which would risk
  double counting via overlapping regional groupings).



In [3]:
print("Loading and Preprocessing Raw Climate Data ")

# Load CSV and Excel files

from google.colab import drive
drive.mount('/content/drive')
save_dir = "/content/drive/MyDrive/climate_risk_data"

temp_anomaly = pd.read_csv('/content/drive/MyDrive/climate_risk_data/temperature-anomaly.csv')
contrib_gas = pd.read_csv('/content/drive/MyDrive/climate_risk_data/contribution-to-temp-rise-by-gas.csv')
contrib_degrees = pd.read_csv('/content/drive/MyDrive/climate_risk_data/contribution-temp-rise-degrees.csv')
gw_land = pd.read_excel('/content/drive/MyDrive/climate_risk_data/global-warming-land.xlsx')
gw_fossil = pd.read_excel('/content/drive/MyDrive/climate_risk_data/global-warming-fossil.xlsx')
ghg_emissions = pd.read_excel('/content/drive/MyDrive/climate_risk_data/ghg-emissions-by-world-region.xlsx')

Loading and Preprocessing Raw Climate Data 
Mounted at /content/drive


# **Filtering**

In [4]:
def to_world_series(df, source_cols, rename_to):
    world = df[df["Entity"].isin(["World", "Global"])].copy()
    world = world[["Year"] + source_cols]
    world.columns = ["Year"] + rename_to
    return world

temp_world = to_world_series(
    temp_anomaly,
    source_cols=["Global average temperature anomaly relative to 1961-1990"],
    rename_to=["temp_anomaly"],
)
gas_world = to_world_series(
    contrib_gas,
    source_cols=[c for c in contrib_gas.columns if c not in ("Entity", "Code", "Year")],
    rename_to=["contrib_n2o", "contrib_methane", "contrib_co2"],
)
degrees_world = to_world_series(
    contrib_degrees,
    source_cols=[c for c in contrib_degrees.columns if c not in ("Entity", "Code", "Year")],
    rename_to=["contrib_total_ghg"],
)
land_world = to_world_series(
    gw_land,
    source_cols=[c for c in gw_land.columns if c not in ("Entity", "Code", "Year")],
    rename_to=["contrib_landuse"],
)
fossil_world = to_world_series(
    gw_fossil,
    source_cols=[c for c in gw_fossil.columns if c not in ("Entity", "Code", "Year")],
    rename_to=["contrib_fossil"],
)
ghg_world = to_world_series(
    ghg_emissions,
    source_cols=[c for c in ghg_emissions.columns if c not in ("Entity", "Code", "Year")],
    rename_to=["ghg_emissions"],
)

# **Inner-join everything on Year so only years present in ALL six datasets**

In [5]:
dfs = [temp_world, gas_world, fossil_world, land_world, ghg_world, degrees_world]
merged = dfs[0]
for d in dfs[1:]:
    merged = merged.merge(d, on="Year", how="inner")

merged = merged.sort_values("Year").reset_index(drop=True)

print(f"Merged dataset shape: {merged.shape}")
print(f"Year range: {merged['Year'].min()}-{merged['Year'].max()}")
print(f"Missing values per column:\n{merged.isna().sum()}")

merged.to_csv(f"{save_dir}/merged_global.csv", index=False)
print(f"\nSaved merged_global.csv to {save_dir}")

Merged dataset shape: (171, 9)
Year range: 1851-2021
Missing values per column:
Year                 0
temp_anomaly         0
contrib_n2o          0
contrib_methane      0
contrib_co2          0
contrib_fossil       0
contrib_landuse      0
ghg_emissions        0
contrib_total_ghg    0
dtype: int64

Saved merged_global.csv to /content/drive/MyDrive/climate_risk_data


# **Exploratory Data Analysis** (EDA)


In [6]:
df = pd.read_csv(f"{save_dir}/merged_global.csv")

#1. Global temperature anomaly over time (Figure 1)
def plot_temperature_anomaly(df):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(df["Year"], df["temp_anomaly"], color="#c0392b", lw=2)
    ax.set_title("Global Temperature Anomaly (1851-2021), relative to 1961-1990 baseline")
    ax.set_xlabel("Year")
    ax.set_ylabel("Temperature anomaly (\u00b0C)")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/eda_temp_anomaly_trend.png", dpi=150)
    plt.show()

#2. Contribution to temperature rise by greenhouse gas (Figure 3)
def plot_gas_contributions(df):
    fig, ax = plt.subplots(figsize=(9, 5))
    for col, label in [("contrib_co2", "CO2"), ("contrib_methane", "Methane"), ("contrib_n2o", "N2O")]:
        ax.plot(df["Year"], df[col], label=label, lw=2)
    ax.set_title("Contribution to Temperature Rise by Greenhouse Gas (World)")
    ax.set_xlabel("Year")
    ax.set_ylabel("Contribution to temp rise (\u00b0C)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/eda_gas_contributions.png", dpi=150)
    plt.show()

#3. Annual global GHG emissions (Figure 2)
def plot_ghg_emissions(df):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(df["Year"], df["ghg_emissions"] / 1e9, color="#2c3e50", lw=2)
    ax.set_title("Annual Global GHG Emissions (CO2-equivalent, billion tonnes)")
    ax.set_xlabel("Year")
    ax.set_ylabel("GHG emissions (Gt CO2e)")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/eda_ghg_emissions.png", dpi=150)
    plt.show()


plot_temperature_anomaly(df)
plot_gas_contributions(df)
plot_ghg_emissions(df)

print(f"Saved charts to {save_dir}")

# Quick summary stats referenced in the report narrative
print(f"\nTemp anomaly: {df['temp_anomaly'].iloc[0]:.2f}\u00b0C ({df['Year'].iloc[0]}) "
      f"-> {df['temp_anomaly'].iloc[-1]:.2f}\u00b0C ({df['Year'].iloc[-1]})")
print(f"GHG emissions: {df['ghg_emissions'].iloc[0]/1e9:.1f} Gt ({df['Year'].iloc[0]}) "
      f"-> {df['ghg_emissions'].iloc[-1]/1e9:.1f} Gt ({df['Year'].iloc[-1]})")

Saved charts to /content/drive/MyDrive/climate_risk_data

Temp anomaly: -0.23°C (1851) -> 0.76°C (2021)
GHG emissions: 4.1 Gt (1851) -> 54.6 Gt (2021)


# **Statistical Correlation & Multicollinearity diagnosis**

    1. Pearson correlation matrix between all candidate warming drivers
       and the temperature anomaly target

    2. Variance Inflation Factor (VIF) for each driver, to formally test
       the brief's expectation that warming drivers are heavily intertwined



In [7]:
df = pd.read_csv(f"{save_dir}/merged_global.csv")

FEATURES = [
    "contrib_n2o", "contrib_methane", "contrib_co2",
    "contrib_fossil", "contrib_landuse", "ghg_emissions", "contrib_total_ghg",
]


def correlation_with_target(df):
    corr = df[FEATURES + ["temp_anomaly"]].corr()
    corr.to_csv(f"{save_dir}/correlation_matrix.csv")

    ranked = corr["temp_anomaly"].drop("temp_anomaly").sort_values(ascending=False)
    print("Correlation of each driver with temp_anomaly:")
    print(ranked)
    return corr


def plot_heatmap(corr):
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr.columns)))
    ax.set_yticklabels(corr.columns)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im)
    ax.set_title("Correlation Matrix: Warming Drivers vs Temperature Anomaly")
    plt.tight_layout()
    plt.savefig(f"{save_dir}/correlation_heatmap.png", dpi=150)
    plt.show()


def compute_vif(df):
    predictor_cols = ["contrib_co2", "contrib_methane", "contrib_n2o",
                       "contrib_fossil", "contrib_landuse", "ghg_emissions"]
    X = df[predictor_cols]

    vif_data = pd.DataFrame()
    vif_data["feature"] = predictor_cols
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(predictor_cols))]

    print("\nVariance Inflation Factors (VIF > 10 conventionally indicates problematic collinearity):")
    print(vif_data.to_string(index=False))
    vif_data.to_csv(f"{save_dir}/vif_results.csv", index=False)
    return vif_data


corr = correlation_with_target(df)
plot_heatmap(corr)
vif_data = compute_vif(df)

print("\nConclusion: pairwise correlations among drivers exceed 0.94 and VIFs are "
      "several orders of magnitude above the VIF > 10 threshold. The contribution-based "
      "variables are statistically derived attributions of the same underlying emissions "
      "series, so ghg_emissions (raw driver, lowest VIF) is retained as the sole regression ")

print(f"\nSaved correlation_matrix.csv, correlation_heatmap.png, vif_results.csv to {save_dir}")


Correlation of each driver with temp_anomaly:
contrib_fossil       0.939317
contrib_co2          0.937555
contrib_n2o          0.936224
contrib_total_ghg    0.933842
contrib_methane      0.919297
ghg_emissions        0.918192
contrib_landuse      0.890753
Name: temp_anomaly, dtype: float64

Variance Inflation Factors (VIF > 10 conventionally indicates problematic collinearity):
        feature          VIF
    contrib_co2 1.797586e+10
contrib_methane 1.616868e+09
    contrib_n2o 1.897184e+07
 contrib_fossil 5.552066e+09
contrib_landuse 1.794369e+09
  ghg_emissions 2.198624e+02

Conclusion: pairwise correlations among drivers exceed 0.94 and VIFs are several orders of magnitude above the VIF > 10 threshold. The contribution-based variables are statistically derived attributions of the same underlying emissions series, so ghg_emissions (raw driver, lowest VIF) is retained as the sole regression 

Saved correlation_matrix.csv, correlation_heatmap.png, vif_results.csv to /content/drive/MyD

## **Predictive Modelling & Actuarial Translation**

    1. Compares four candidate baseline regression specifications, trained
       on 1950-2005 and validated on the unseen 2006-2021 period, using
       MAE / RMSE / R^2

    2. Selects the best-performing specification (GHG emissions, linear).

    3. Produces the actual-vs-predicted test-set chart

    4. Projects the temperature anomaly ten years forward to 2031



In [8]:
df = pd.read_csv(f"{save_dir}/merged_global.csv")

TRAIN_START, TRAIN_END = 1950, 2005
TEST_START = 2006

train = df[(df["Year"] >= TRAIN_START) & (df["Year"] <= TRAIN_END)]
test = df[df["Year"] >= TEST_START]
y_train, y_test = train["temp_anomaly"].values, test["temp_anomaly"].values


def evaluate(name, model, X_train, y_train, X_test, y_test, results):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    results.append({"model": name, "MAE": mae, "RMSE": rmse, "R2": r2})
    return model, pred


In [9]:
# Compare four candidate specifications

results = []

# 1. Year only (simple linear time trend)
evaluate("Year (linear trend)", LinearRegression(),
          train[["Year"]], y_train, test[["Year"]], y_test, results)

# 2. GHG emissions only (linear) ,selected as final model
ghg_model, _ = evaluate("GHG emissions (linear)", LinearRegression(),
          train[["ghg_emissions"]], y_train, test[["ghg_emissions"]], y_test, results)

# 3. GHG emissions (quadratic)
poly = PolynomialFeatures(degree=2, include_bias=False)
Xtr_poly = poly.fit_transform(train[["ghg_emissions"]])
Xte_poly = poly.transform(test[["ghg_emissions"]])
evaluate("GHG emissions (quadratic)", LinearRegression(),
          Xtr_poly, y_train, Xte_poly, y_test, results)

# 4. Year + GHG emissions combined
evaluate("Year + GHG emissions", LinearRegression(),
          train[["Year", "ghg_emissions"]], y_train,
          test[["Year", "ghg_emissions"]], y_test, results)

results_df = pd.DataFrame(results)
results_df.to_csv(f"{save_dir}/model_comparison.csv", index=False)
print("Candidate model comparison (test set 2006-2021):")
print(results_df.to_string(index=False))

print(f"\nSelected final model: GHG emissions (linear) ,lowest MAE/RMSE of all candidates.")
print(f"temp_anomaly = {ghg_model.intercept_:.4f} + {ghg_model.coef_[0]:.4e} * ghg_emissions")
print("\nNote: R2 is negative for every candidate model. This reflects genuine post-2005 "
      "acceleration in warming relative to the 1950-2005 fitted trend  "
      "not a coding error,the model's absolute errors (MAE/RMSE) remain modest (<0.2C) but "
      "it systematically underestimates the test period.")

Candidate model comparison (test set 2006-2021):
                    model      MAE     RMSE        R2
      Year (linear trend) 0.165658 0.193751 -0.839706
   GHG emissions (linear) 0.132141 0.162045 -0.286850
GHG emissions (quadratic) 0.380970 0.404022 -6.999601
     Year + GHG emissions 0.224912 0.244624 -1.932631

Selected final model: GHG emissions (linear) ,lowest MAE/RMSE of all candidates.
temp_anomaly = -0.6423 + 2.3743e-11 * ghg_emissions

Note: R2 is negative for every candidate model. This reflects genuine post-2005 acceleration in warming relative to the 1950-2005 fitted trend  not a coding error,the model's absolute errors (MAE/RMSE) remain modest (<0.2C) but it systematically underestimates the test period.


In [10]:
# Actual vs predicted on the test set (Figure 5)

pred_test = ghg_model.predict(test[["ghg_emissions"]])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df["Year"], df["temp_anomaly"], color="gray", alpha=0.4, label="Full historical series")
ax.plot(train["Year"], y_train, color="#2980b9", lw=2, label="Train (1950-2005)")
ax.plot(test["Year"], y_test, color="#c0392b", lw=2, label="Test actual (2006-2021)")
ax.plot(test["Year"], pred_test, color="#c0392b", lw=2, ls="--", label="Test predicted")
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlabel("Year")
ax.set_ylabel("Temperature anomaly (\u00b0C)")
ax.set_title("Baseline Regression: Actual vs Predicted Temperature Anomaly (Test Set)")
plt.tight_layout()
plt.savefig(f"{save_dir}/model_test_fit.png", dpi=150)
plt.show()

In [11]:
# 10-year projection (Figure 6)

recent = df[df["Year"] >= 2000]
emissions_trend = LinearRegression().fit(recent[["Year"]], recent["ghg_emissions"])

future_years = np.arange(2022, 2032)
future_emissions = emissions_trend.predict(future_years.reshape(-1, 1))
future_temp = ghg_model.predict(future_emissions.reshape(-1, 1))

proj = pd.DataFrame({
    "Year": future_years,
    "proj_ghg_emissions": future_emissions,
    "proj_temp_anomaly": future_temp,
})
proj.to_csv(f"{save_dir}/projection_2022_2031.csv", index=False)
print("\n10-year projection:")
print(proj.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df["Year"], df["temp_anomaly"], color="#2c3e50", lw=2, label="Historical (1851-2021)")
ax.plot(future_years, future_temp, color="#e67e22", lw=2, ls="--", marker="o",
        label="10-year projection (2022-2031)")
ax.axvline(2021, color="gray", ls=":", alpha=0.5)
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlabel("Year")
ax.set_ylabel("Temperature anomaly (\u00b0C)")
ax.set_title("10-Year Temperature Anomaly Projection for Aethelgard Re Risk Modelling")
plt.tight_layout()
plt.savefig(f"{save_dir}/projection_chart.png", dpi=150)
plt.show()

print(f"\nSaved model_comparison.csv, model_test_fit.png, projection_2022_2031.csv, "
      f"projection_chart.png to {save_dir}")



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(



10-year projection:
 Year  proj_ghg_emissions  proj_temp_anomaly
 2022        5.703256e+10           0.711804
 2023        5.768859e+10           0.727380
 2024        5.834462e+10           0.742956
 2025        5.900065e+10           0.758533
 2026        5.965669e+10           0.774109
 2027        6.031272e+10           0.789685
 2028        6.096875e+10           0.805261
 2029        6.162478e+10           0.820837
 2030        6.228081e+10           0.836414
 2031        6.293685e+10           0.851990

Saved model_comparison.csv, model_test_fit.png, projection_2022_2031.csv, projection_chart.png to /content/drive/MyDrive/climate_risk_data
